# Can you publish the halftime score?

A report editor asks for the halftime score of fictional match **M17**, Harbor vs Vale, and the recorded settlement of ticket **T8** on Harbor leading at halftime. Both rows belong in the report. Each needs its own evidence.

You have three deadlines on the same invented match day. At each deadline, write down the score and the ticket status you can support. Cite the record ID beside each claim. If a field is missing, name the record you still need. All times below are UTC.

**Run the exercise.** In Basthon choose **Cell → Run All**. In Jupyter choose **Run → Run All Cells**. Change the deadline in the report cell and run it again. The notebook includes every fictional record and uses Python's standard library.

## Read the desk packet

| Received | Record | What reached the desk |
|---|---|---|
| 18:46 | S1 | M17 ended regulation 3–1. Harbor is listed first. |
| 18:47 | S2 | M71 reached halftime at 0–2. It is another fixture. |
| 18:48 | G1 | The example book graded ticket T80 as lost. It belongs to another selection record. |
| 18:49 | G2 | The example book lists T8 as pending. |
| 18:55 | S3 | The scorer confirms the M17 halftime score was 1–0. |
| 19:05 | G3 | The example book confirms T8 was graded won. |

The scoring record joins on **match + period**. The grading record joins on **book + ticket + match + period + selection**. A received time tells you when the editor could use a record. IDs beginning with S come from the scorer. IDs beginning with G come from the example book.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Record:
    received: str
    source_id: str
    subject: tuple
    value: str

score_subject = ("official scorer", "M17", "first half")
ticket_subject = ("example book", "T8", "M17", "first half", "Harbor leads")

records = [
    Record("18:46", "S1", ("official scorer", "M17", "regulation"), "3–1"),
    Record("18:47", "S2", ("official scorer", "M71", "first half"), "0–2"),
    Record("18:48", "G1", ("example book", "T80", "M17", "first half", "Vale leads"), "lost"),
    Record("18:49", "G2", ticket_subject, "pending"),
    Record("18:55", "S3", score_subject, "1–0"),
    Record("19:05", "G3", ticket_subject, "won"),
]

## Make an editorial decision

Start at **18:50**. Which match and period does S1 identify? Could S2 fill the gap? Decide whether G2 supplies a final grade. Now advance the clock to **18:55** and then **19:05**. For each change, say which source arrived and which sentence in your report can change.

The function uses only records received by the selected deadline. An exact-subject match with a pending grade is evidence that the book reported pending. The grade itself is still unresolved.

In [ ]:
from datetime import time

def latest_record(packet, subject, deadline):
    cutoff = time.fromisoformat(deadline)
    matches = [r for r in packet if r.subject == subject and time.fromisoformat(r.received) <= cutoff]
    if not matches:
        return "unknown", "no matching record"
    latest = max(time.fromisoformat(r.received) for r in matches)
    current = [r for r in matches if time.fromisoformat(r.received) == latest]
    sources = ", ".join(sorted(r.source_id for r in current))
    if len({r.value for r in current}) != 1:
        return "conflicting records", sources
    return current[0].value, sources

def report(deadline):
    return {
        "M17 first-half score": latest_record(records, score_subject, deadline),
        "T8 reported settlement": latest_record(records, ticket_subject, deadline),
    }

In [ ]:
deadline = "18:50"  # Try "18:55" or "19:05".
for claim, (value, evidence) in report(deadline).items():
    print(f"{deadline} UTC | {claim} | {value} | {evidence}")

## Check your report

| Deadline | M17 first-half score | T8 reported settlement | Editor's next action |
|---|---|---|---|
| 18:50 | Unknown. S1 has the wrong period and S2 has the wrong match. | Pending, G2. Final grade unknown. | Request the M17 first-half scoring record and wait for the book's T8 grade. |
| 18:55 | 1–0, S3. | Pending, G2. | Publish the score with its source. Retain the grade as pending. |
| 19:05 | 1–0, S3. | Won, G3. | Add the grade and cite the book confirmation. |

**Hand off the report.** Give the next editor the deadline, fixture, period, ticket and selection. For each report field include its value, source ID and any missing evidence. A report can correctly record an unknown. Substituting 3–1 at halftime or treating S3 as the book's grade fails the assignment.

**Try one more change.** Suppose a scorer sends a different M17 halftime score at the same received time as S3. Run the last cell. Which source IDs would you take back to the scorer for reconciliation?

In [ ]:
disputed = records + [Record("18:55", "S4", score_subject, "0–0")]
value, sources = latest_record(disputed, score_subject, "18:55")
print(f"Correction desk | M17 first-half score | {value} | {sources}")

The match, book and ticket in this lesson are fictional. For published match data, open the [World Cup score explorer](https://github.com/JacobiusMakes/parlayapi-notebooks/tree/main/labs/world-cup-scorelines). Its tournament records distinguish regulation, cumulative extra-time goals and shoot-outs. The report exercise adds fictional halftime and ticket records for a different task.